# 001a — Prepare Train Data (Expert Labels)

訓練資料來源：`diagnosis_1` ～ `diagnosis_4`（肺科醫師標註）

Pipeline：
1. 過濾 `cough_detected > 0.8`
2. 用 `expert_label`（已內建多數決，平票為 NaN）篩選有效樣本
3. 五類專家標籤 → 三分類（`covid` / `symptomatic` / `healthy`）
4. `symptomatic` 內部**不做** balancing（保留全部 1403 筆，增加樣本數）
5. 三分類對 `covid` 和 `healthy` 做 balancing（對齊 `symptomatic` 數量或取最小值）
6. 提取音頻特徵，輸出 `prepared_train_coughvid_balanced.csv`

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
from sklearn.utils import resample
from utils import preproces

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 設定路徑

In [2]:
COUGHVID_ROOT = 'C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav'
METADATA_CSV  = 'C:/Users/aint/Desktop/RNN/Final_project/metadata_compiled.csv'
COUGH_DETECTED_THRESHOLD = 0.8
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 讀取 metadata

In [3]:
df_meta = pd.read_csv(METADATA_CSV)
print('原始樣本數:', len(df_meta))

df_meta = df_meta[df_meta['cough_detected'] > COUGH_DETECTED_THRESHOLD].copy()
print('cough_detected 過濾後:', len(df_meta))

原始樣本數: 34434
cough_detected 過濾後: 18541


## 篩選有效專家標註樣本

In [4]:
VALID_DIAGNOSES = {'COVID-19', 'healthy_cough', 'upper_infection', 'lower_infection', 'obstructive_disease'}
EXPERT_COLS     = ['diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4']

def majority_vote(row):
    votes = [row[col] for col in EXPERT_COLS if pd.notna(row.get(col)) and row[col] in VALID_DIAGNOSES]
    if not votes:
        return None
    count = Counter(votes)
    top = count.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return None  # 平票丟棄
    return top[0][0]

df_meta['expert_label_voted'] = df_meta.apply(majority_vote, axis=1)
df_expert = df_meta[df_meta['expert_label_voted'].isin(VALID_DIAGNOSES)].copy()
df_expert = df_expert.rename(columns={'expert_label_voted': 'expert_label'})

print('有效專家標註樣本數:', len(df_expert))
print('expert_label 分佈:')
print(df_expert['expert_label'].value_counts())

有效專家標註樣本數: 2416
expert_label 分佈:
expert_label
upper_infection        681
lower_infection        561
healthy_cough          543
COVID-19               470
obstructive_disease    161
Name: count, dtype: int64


## 五類 → 三分類對應

```
COVID-19             → covid
healthy_cough        → healthy
upper_infection   ┐
lower_infection   ├→ symptomatic
obstructive_disease┘
```

In [5]:
LABEL_MAP = {
    'COVID-19':            'covid',
    'healthy_cough':       'healthy',
    'upper_infection':     'symptomatic',
    'lower_infection':     'symptomatic',
    'obstructive_disease': 'symptomatic',
}
df_expert['label'] = df_expert['expert_label'].map(LABEL_MAP)

print('三分類分佈:')
print(df_expert['label'].value_counts())
print()
print('symptomatic 子類分佈:')
print(df_expert[df_expert['label'] == 'symptomatic']['expert_label'].value_counts())

三分類分佈:
label
symptomatic    1403
healthy         543
covid           470
Name: count, dtype: int64

symptomatic 子類分佈:
expert_label
upper_infection        681
lower_infection        561
obstructive_disease    161
Name: count, dtype: int64


## Balancing

`symptomatic` 內部**不做** balancing，直接保留全部樣本以增加資料量。
三分類以最少的類別（`covid`）為基準做 downsampling。

In [6]:
df_covid   = df_expert[df_expert['label'] == 'covid']
df_healthy = df_expert[df_expert['label'] == 'healthy']
df_symp    = df_expert[df_expert['label'] == 'symptomatic']

three_class_min = min(len(df_covid), len(df_healthy), len(df_symp))
print(f'各類樣本數 — covid: {len(df_covid)}, healthy: {len(df_healthy)}, symptomatic: {len(df_symp)}')
print(f'Balancing 目標（最小值）: {three_class_min}')

df_balanced = pd.concat([
    resample(df_covid,   replace=False, n_samples=three_class_min, random_state=42),
    resample(df_healthy, replace=False, n_samples=three_class_min, random_state=42),
    resample(df_symp,    replace=False, n_samples=three_class_min, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print('\nBalancing 後三分類分佈:')
print(df_balanced['label'].value_counts())
print('總樣本數:', len(df_balanced))

各類樣本數 — covid: 470, healthy: 543, symptomatic: 1403
Balancing 目標（最小值）: 470

Balancing 後三分類分佈:
label
covid          470
healthy        470
symptomatic    470
Name: count, dtype: int64
總樣本數: 1410


## 確認 .wav 檔案存在

In [7]:
def find_wav(uuid):
    path = os.path.join(COUGHVID_ROOT, f'{uuid}.wav')
    return path if os.path.exists(path) else None

df_balanced['file_path'] = df_balanced['uuid'].apply(find_wav)
missing = df_balanced['file_path'].isna().sum()
print(f'找不到 .wav 的樣本數: {missing}（將被移除）')
df_balanced = df_balanced.dropna(subset=['file_path']).reset_index(drop=True)
print(f'最終可用樣本數: {len(df_balanced)}')

找不到 .wav 的樣本數: 0（將被移除）
最終可用樣本數: 1410


## 特徵提取

In [8]:
FEATURE_COLS = [
    'filename', 'chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth',
    'rolloff', 'zero_crossing_rate',
    *[f'mfcc{i}' for i in range(1, 21)],
    'label'
]

feature_rows = []
errors = []

for _, row in tqdm(df_balanced.iterrows(), total=len(df_balanced)):
    try:
        feat = preproces(row['file_path'])
        if isinstance(feat, pd.Series):
            feat = feat.to_dict()
        feat['filename'] = os.path.basename(row['file_path'])
        feat['label']    = row['label']
        feature_rows.append(feat)
    except Exception as e:
        errors.append({'file': row['file_path'], 'error': str(e)})

print(f'成功: {len(feature_rows)} 筆，失敗: {len(errors)} 筆')

df_features = pd.DataFrame(feature_rows).reindex(columns=FEATURE_COLS)
out_path = os.path.join(OUTPUT_DIR, 'prepared_train_coughvid_balanced.csv')
df_features.to_csv(out_path, index=False)
print(f'\n已儲存至 {out_path}')
print(df_features['label'].value_counts())
print('總樣本數:', len(df_features))
df_features.head()

 61%|██████    | 857/1410 [00:32<00:20, 26.60it/s]c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 1410/1410 [00:53<00:00, 26.48it/s]

成功: 1410 筆，失敗: 0 筆

已儲存至 data\prepared_train_coughvid_balanced.csv
label
covid          470
healthy        470
symptomatic    470
Name: count, dtype: int64
總樣本數: 1410


,filename,chroma_stft,rmse,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,mfcc1,mfcc2,mfcc3,...,mfcc12,mfcc13,mfcc14,mfcc15,mfcc16,mfcc17,mfcc18,mfcc19,mfcc20,label
0,99524a84-e055-422e-831e-f22188948c84.wav,0.218001,0.046092,1223.814308,866.046312,2143.949382,0.077393,-454.239380,38.458252,-14.110901,...,-6.642495,0.514142,-4.662922,-1.512830,-1.292310,-4.308308,-1.133466,0.423705,-0.717857,covid
1,7b5698cb-b713-4f6d-bb5f-6142b6f509a3.wav,0.427838,0.041234,1756.702840,1246.849652,3084.999151,0.135951,-387.894653,57.112675,-44.717087,...,-14.854997,-0.696939,-7.807527,-5.253421,-6.597958,-3.449753,-6.489015,-2.253315,-3.805197,healthy
2,dec024bb-bb3f-49ba-8a42-b1420d56a2e0.wav,0.585251,0.030173,2529.497993,2077.287551,5137.313843,0.179620,-457.737640,35.949787,-17.002607,...,-1.886410,4.739943,-3.758386,2.082281,-4.971026,-0.995281,-3.388829,-2.314761,-3.605460,symptomatic
3,8ac2ef46-05fb-4f68-bb2d-ca5b1e51ed7f.wav,0.518158,0.082573,2434.272627,1896.540916,4698.126221,0.166958,-305.935822,77.755081,-37.934708,...,-16.333807,4.203900,-11.738227,1.581684,-5.374125,-3.866661,-4.724221,-5.321432,-3.139173,covid
4,dcd085b9-d8d4-4c6f-8082-0214e523cdba.wav,0.572435,0.021823,2439.967516,2164.585576,5203.895805,0.179742,-529.451965,35.761845,-2.521456,...,-3.525230,1.678645,-3.156752,1.462249,-4.329421,-0.451444,-1.735319,-0.968039,-0.262841,healthy


## 失敗檔案檢查（選用）

In [9]:
if errors:
    print(pd.DataFrame(errors).to_string())
else:
    print('沒有失敗的檔案！')

沒有失敗的檔案！
